[Énoncés : Série 5 — Graphes, BFS et DFS](../serie5_graphes_bfs_dfs.ipynb)

# Corrigé — Série 5 — Graphes, BFS et DFS

Pour chaque exercice : l'idée, la complexité, puis une solution commentée avec ses tests. Il existe souvent plusieurs bonnes solutions : si la tienne passe tous les tests et le test de vitesse, elle est juste. Compare surtout l'**idée**.

Exécute d'abord la cellule d'outils.

In [ ]:
# Outils de ce notebook : exécute cette cellule en premier.
# - verifier(fonction, cas) lance ta fonction sur chaque cas et affiche ✅ (bon), ❌ (mauvais) ou 💥 (plantage).
# - chrono(fonction, ...) mesure le temps sur une grande entrée (tests de vitesse facultatifs).
# Tu n'as pas besoin de comprendre ce code pour t'en servir.

import copy
import time

def _court(texte, limite=100):
    return texte if len(texte) <= limite else texte[:limite - 3] + "..."

def verifier(fonction, cas, ne_pas_modifier=False):
    reussis = 0
    for arguments, attendu in cas:
        texte_args = _court(", ".join(repr(a) for a in arguments))
        copie = copy.deepcopy(arguments)
        try:
            obtenu = fonction(*copie)
        except Exception as erreur:
            print(f"💥 {fonction.__name__}({texte_args}) a planté : {type(erreur).__name__}: {erreur}")
            continue
        if obtenu != attendu:
            print(f"❌ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
        elif ne_pas_modifier and copie != arguments:
            print(f"⚠️ {fonction.__name__}({texte_args}) -> bon résultat, mais ta fonction a modifié son entrée (interdit ici)")
        else:
            reussis += 1
            print(f"✅ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}")
    print(f"\n{reussis}/{len(cas)} tests réussis")

def chrono(fonction, *arguments, attendu=None, limite=1.0):
    debut = time.perf_counter()
    try:
        obtenu = fonction(*arguments)
    except Exception as erreur:
        print(f"💥 {fonction.__name__} a planté : {type(erreur).__name__}: {erreur}")
        return
    duree = time.perf_counter() - debut
    if obtenu is None:
        print("⏳ Ta fonction ne renvoie rien pour l'instant : écris-la d'abord.")
    elif attendu is not None and obtenu != attendu:
        print(f"❌ Résultat faux sur la grande entrée : {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
    elif duree <= limite:
        print(f"✅ {duree:.3f} s : assez rapide")
    else:
        print(f"⚠️ {duree:.1f} s : trop lent (limite : {limite} s). Ta solution n'a probablement pas la complexité attendue.")

## Exercice 24 — Salles accessibles depuis l'entrée

**Idée** : une exploration (DFS ou BFS, thème 5) depuis `S`. Le set `visites` contient exactement les cases atteintes : la réponse est sa taille.

**Complexité** : O(lignes × colonnes).

In [ ]:
def salles_accessibles(plan: list[str]) -> int:
    lignes, colonnes = len(plan), len(plan[0])
    for i in range(lignes):                       # trouver S
        for j in range(colonnes):
            if plan[i][j] == "S":
                depart = (i, j)

    visites = {depart}
    pile = [depart]
    while pile:                                   # DFS avec une pile (BFS marcherait aussi)
        i, j = pile.pop()
        for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            ni, nj = i + di, j + dj
            if (0 <= ni < lignes and 0 <= nj < colonnes
                    and plan[ni][nj] != "#" and (ni, nj) not in visites):
                visites.add((ni, nj))
                pile.append((ni, nj))
    return len(visites)


verifier(salles_accessibles, [
    ((['S..', '#.#', '...'],), 7),
    ((['S'],), 1),
    ((['S#.'],), 1),   # la case libre est derrière un mur
    ((['S.#.', '..#.', '###.'],), 4),
    ((['#.#', '#S#', '###'],), 2),
    ((['S...', '####', '....'],), 4),
])

## Exercice 25 — Combien de réseaux séparés ?

**Idée** : exactement `nombre_iles` du thème 5, sur un graphe au lieu d'une grille. On construit la liste d'adjacence, puis chaque sommet pas encore visité démarre un nouveau réseau, qu'on explore entièrement pour le marquer.

**Complexité** : O(n + nombre de liens).

In [ ]:
def nombre_reseaux(n: int, liens: list[list[int]]) -> int:
    voisins = [[] for _ in range(n)]          # liste d'adjacence
    for a, b in liens:
        voisins[a].append(b)
        voisins[b].append(a)

    visites = set()
    reseaux = 0
    for depart in range(n):
        if depart not in visites:
            reseaux += 1                      # jamais atteint : c'est un nouveau réseau
            visites.add(depart)
            pile = [depart]
            while pile:                       # on marque tout ce réseau
                s = pile.pop()
                for v in voisins[s]:
                    if v not in visites:
                        visites.add(v)
                        pile.append(v)
    return reseaux


verifier(nombre_reseaux, [
    ((6, [[0, 1], [1, 2], [3, 4]]), 3),
    ((1, []), 1),
    ((4, []), 4),   # que des ordinateurs isolés
    ((3, [[0, 1], [1, 2], [2, 0]]), 1),
    ((5, [[0, 4], [1, 3]]), 3),
    ((0, []), 0),
])

## Exercice 26 — Distance depuis un sommet

**Idée** : le **BFS** du thème 5, qui calcule justement les distances dans un graphe sans poids. La liste `distance` sert aussi de « visités » : une case encore à `-1` n'a pas été atteinte.

**Complexité** : O(n + nombre de liens). Un DFS ne donnerait pas les distances minimales.

In [ ]:
from collections import deque

def distances_reseau(n: int, liens: list[list[int]], depart: int) -> list[int]:
    voisins = [[] for _ in range(n)]
    for a, b in liens:
        voisins[a].append(b)
        voisins[b].append(a)

    distance = [-1] * n              # -1 = pas (encore) atteint
    distance[depart] = 0
    file = deque([depart])
    while file:
        s = file.popleft()
        for v in voisins[s]:
            if distance[v] == -1:    # la première fois qu'on l'atteint, c'est par un plus court chemin
                distance[v] = distance[s] + 1
                file.append(v)
    return distance


verifier(distances_reseau, [
    ((5, [[0, 1], [1, 2], [0, 3]], 0), [0, 1, 2, 1, -1]),
    ((1, [], 0), [0]),
    ((3, [], 1), [-1, 0, -1]),
    ((4, [[0, 1], [1, 2], [2, 3], [3, 0]], 0), [0, 1, 2, 1]),
    ((5, [[0, 1], [1, 2], [2, 3], [3, 4], [0, 4]], 2), [2, 1, 0, 1, 2]),
    ((4, [[0, 1], [0, 2], [1, 3], [2, 3]], 3), [2, 1, 1, 0]),
])

## Exercice 27 — Un cycle existe-t-il dans le réseau ?

**Idée** : une exploration qui retient, pour chaque sommet, **par où on est arrivé** (le parent). Si l'on retombe sur un sommet déjà visité autrement qu'en revenant vers le parent, c'est qu'il existe deux chemins différents vers lui : il y a un cycle. Ne pas oublier les graphes en plusieurs morceaux.

Autre raisonnement, sans exploration : un groupe connecté de `s` sommets sans cycle a exactement `s - 1` liens.

**Complexité** : O(n + nombre de liens).

In [ ]:
def contient_cycle(n: int, liens: list[list[int]]) -> bool:
    voisins = [[] for _ in range(n)]
    for a, b in liens:
        voisins[a].append(b)
        voisins[b].append(a)

    visites = set()
    for depart in range(n):                   # le graphe peut avoir plusieurs groupes
        if depart in visites:
            continue
        visites.add(depart)
        pile = [(depart, -1)]                 # (sommet, sommet par lequel on est arrivé)
        while pile:
            s, parent = pile.pop()
            for v in voisins[s]:
                if v == parent:
                    continue                  # reprendre le lien par lequel on vient : pas un cycle
                if v in visites:
                    return True               # déjà atteint par un autre chemin : il y a un cycle
                visites.add(v)
                pile.append((v, s))
    return False


verifier(contient_cycle, [
    ((3, [[0, 1], [1, 2], [2, 0]]), True),
    ((4, [[0, 1], [1, 2], [2, 3]]), False),
    ((1, []), False),
    ((5, [[0, 1], [2, 3], [3, 4], [4, 2]]), True),   # le cycle est dans un deuxième groupe
    ((4, [[0, 1], [0, 2], [0, 3]]), False),   # une étoile n'a pas de cycle
    ((4, [[0, 1], [1, 2], [2, 3], [3, 1]]), True),
    ((6, [[0, 1], [1, 2], [3, 4]]), False),
])

## Exercice 28 — Séparer le réseau en deux équipes

**Idée** : on « colorie » en explorant. Le premier sommet d'un groupe va en A ; chaque voisin est **forcé** dans l'autre équipe. Si l'on trouve un lien dont les deux extrémités ont déjà la même équipe, c'est impossible. On recommence pour chaque groupe non encore colorié.

**Complexité** : O(n + nombre de liens).

In [ ]:
def est_biparti(n: int, liens: list[list[int]]) -> bool:
    voisins = [[] for _ in range(n)]
    for a, b in liens:
        voisins[a].append(b)
        voisins[b].append(a)

    equipe = [None] * n                     # "A", "B" ou None (pas encore placé)
    for depart in range(n):
        if equipe[depart] is not None:
            continue
        equipe[depart] = "A"                # premier sommet d'un groupe : équipe au choix
        pile = [depart]
        while pile:
            s = pile.pop()
            autre = "B" if equipe[s] == "A" else "A"
            for v in voisins[s]:
                if equipe[v] is None:
                    equipe[v] = autre       # un voisin va forcément dans l'autre équipe
                    pile.append(v)
                elif equipe[v] == equipe[s]:
                    return False            # deux voisins dans la même équipe : impossible
    return True


verifier(est_biparti, [
    ((4, [[0, 1], [1, 2], [2, 3], [3, 0]]), True),
    ((3, [[0, 1], [1, 2], [2, 0]]), False),
    ((1, []), True),
    ((5, []), True),   # aucun lien
    ((6, [[0, 1], [2, 3], [3, 4], [4, 2]]), False),   # le triangle est dans un deuxième groupe
    ((5, [[0, 1], [1, 2], [2, 3], [3, 4]]), True),
    ((5, [[0, 1], [1, 2], [2, 3], [3, 4], [4, 0]]), False),   # cycle de 5 sommets
])

## Exercice 29 — Minutes de propagation

**Idée** : un **BFS à plusieurs départs** (le lien « contamination = BFS » du thème 5). On met toutes les cases contaminées dans la file dès le début, à la minute 0. Chaque couche du BFS correspond à une minute ; la réponse est la minute de la dernière case atteinte. On compte les cases saines pour détecter celles qui ne sont jamais atteintes.

**Complexité** : O(lignes × colonnes). Simuler minute par minute en reparcourant toute la grille coûterait O((lignes × colonnes)²) dans le pire cas.

In [ ]:
from collections import deque

def minutes_propagation(grille: list[list[int]]) -> int:
    lignes, colonnes = len(grille), len(grille[0])
    file = deque()
    sains = 0
    for i in range(lignes):
        for j in range(colonnes):
            if grille[i][j] == 2:
                file.append((i, j, 0))      # TOUTES les sources démarrent en même temps (minute 0)
            elif grille[i][j] == 1:
                sains += 1

    deja = {(i, j) for i, j, _ in file}
    minutes = 0
    while file:
        i, j, m = file.popleft()
        minutes = max(minutes, m)
        for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            ni, nj = i + di, j + dj
            if (0 <= ni < lignes and 0 <= nj < colonnes
                    and grille[ni][nj] == 1 and (ni, nj) not in deja):
                deja.add((ni, nj))
                sains -= 1
                file.append((ni, nj, m + 1))    # contaminée une minute plus tard
    return minutes if sains == 0 else -1


verifier(minutes_propagation, [
    (([[2, 1, 1], [1, 1, 0], [0, 1, 1]],), 4),
    (([[2, 1, 1], [0, 1, 1], [1, 0, 1]],), -1),   # le 1 en bas à gauche est isolé
    (([[0, 2]],), 0),   # aucune case saine
    (([[0]],), 0),
    (([[1]],), -1),   # aucune source de contamination
    (([[2, 1, 1, 1]],), 3),
    (([[2, 1, 2]],), 1),   # deux sources en même temps
    (([[1, 1, 1], [1, 2, 1], [1, 1, 1]],), 2),
])

---

[Énoncés de la série](../serie5_graphes_bfs_dfs.ipynb)